In [11]:
import pandas as pd
import numpy as np
df = pd.DataFrame({
    'player': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', None],
    'age': [24, 28, np.nan, 24, 30, 22],
    'goals': [10, 15, 8, 10, None, 7],
    'contract_years': [2, 1, 3, 2, None, 2]
})

print("Original Data:")
print(df)
df = df.drop_duplicates()
print("\nDuplicates removed:")
df['age'] = df['age'].fillna(df['age'].median())
df['goals'] = df['goals'].fillna(df['goals'].median())
df['player'] = df['player'].fillna(df['player'].mode()[0])
df['contract_years'] = df['contract_years'].fillna(df['contract_years'].median())
print(df)


Original Data:
    player   age  goals  contract_years
0    Alice  24.0   10.0             2.0
1      Bob  28.0   15.0             1.0
2  Charlie   NaN    8.0             3.0
3    Alice  24.0   10.0             2.0
4    David  30.0    NaN             NaN
5     None  22.0    7.0             2.0

Duplicates removed:
    player   age  goals  contract_years
0    Alice  24.0   10.0             2.0
1      Bob  28.0   15.0             1.0
2  Charlie  26.0    8.0             3.0
4    David  30.0    9.0             2.0
5    Alice  22.0    7.0             2.0


In [12]:
# Feature Engineering
df['goals_past_3_seasons'] = [7, 13, 5, 7, 9]
df['injuries'] = [1, 0, 3, 1, 2]
df['games_played'] = [25, 34, 22, 25, 30]
df['goal_trend'] = (df['goals'] - df['goals_past_3_seasons']) / df['goals_past_3_seasons']
df['injury_risk'] = df['injuries'] / df['games_played']
df['contract_left'] = df['contract_years'] - 1

print(df[['player', 'goals', 'goals_past_3_seasons', 'goal_trend', 'injury_risk', 'contract_left']])

    player  goals  goals_past_3_seasons  goal_trend  injury_risk  \
0    Alice   10.0                     7    0.428571     0.040000   
1      Bob   15.0                    13    0.153846     0.000000   
2  Charlie    8.0                     5    0.600000     0.136364   
4    David    9.0                     7    0.285714     0.040000   
5    Alice    7.0                     9   -0.222222     0.066667   

   contract_left  
0            1.0  
1            0.0  
2            2.0  
4            1.0  
5            1.0  


In [7]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
scaler = MinMaxScaler()
num_cols = ['age', 'goals', 'games_played', 'contract_left']
df[num_cols] = scaler.fit_transform(df[num_cols])

encoder = OneHotEncoder()
cat_array = encoder.fit_transform(df[['player']]).toarray()
cat_columns = encoder.get_feature_names_out(['player'])
cat_df = pd.DataFrame(cat_array, columns=cat_columns)
df = pd.concat([df.reset_index(drop=True), cat_df], axis=1)

print(df.head())

    player   age  goals  contract_years  goals_past_3_seasons  injuries  \
0    Alice  0.25  0.375             2.0                     7         1   
1      Bob  0.75  1.000             1.0                    13         0   
2  Charlie  0.50  0.125             3.0                     5         3   
3    David  1.00  0.250             2.0                     7         1   
4    Alice  0.00  0.000             2.0                     9         2   

   games_played  goal_trend  injury_risk  contract_left  player_Alice  \
0      0.250000    0.428571     0.040000            0.5           1.0   
1      1.000000    0.153846     0.000000            0.0           0.0   
2      0.000000    0.600000     0.136364            1.0           0.0   
3      0.250000    0.285714     0.040000            0.5           0.0   
4      0.666667   -0.222222     0.066667            0.5           1.0   

   player_Bob  player_Charlie  player_David  
0         0.0             0.0           0.0  
1         1.0     

In [8]:
from textblob import TextBlob
tweets = [
    "Alice's performance was excellent today!",
    "Bob was off his game, seemed distracted.",
    "Charlie back from injury--solid match.",
    "David's contract negotiations affecting focus.",
    "Alice continues to impress season after season.",
]
sent_df = pd.DataFrame({'player': ['Alice','Bob','Charlie','David','Alice'], 'tweet': tweets})
sent_df['polarity'] = sent_df['tweet'].apply(lambda x: TextBlob(x).sentiment.polarity)
sent_df['sentiment'] = sent_df['polarity'].apply(lambda x: 'Positive' if x > 0 else ('Negative' if x < 0 else 'Neutral'))
print(sent_df)
print("\nSentiment summary:")
print(sent_df['sentiment'].value_counts())


    player                                            tweet  polarity  \
0    Alice         Alice's performance was excellent today!       1.0   
1      Bob         Bob was off his game, seemed distracted.      -0.4   
2  Charlie           Charlie back from injury--solid match.       0.0   
3    David   David's contract negotiations affecting focus.       0.0   
4    Alice  Alice continues to impress season after season.       0.0   

  sentiment  
0  Positive  
1  Negative  
2   Neutral  
3   Neutral  
4   Neutral  

Sentiment summary:
sentiment
Neutral     3
Positive    1
Negative    1
Name: count, dtype: int64
